# Gene Function Lab — Pipeline Notebook

This notebook populates and maintains the gene function database.
The website at **huggingface.co/spaces/shenovad/reyalab** reads from this DB automatically.

---

## Setup — run once per session in order

| Cell | What it does |
|------|--------------|
| Cell 1 | Install dependencies |
| Cell 2 | Mount Drive + authenticate |
| Cell 3 | Configure paths + verify DB |

## Actions — run as needed

| Cell | What it does | When |
|------|-------------|------|
| Cell 4 | Add a list of genes to DB | Anytime you have new genes |
| Cell 5 | Process website gene requests | When lab members request genes |
| Cell 6 | Refresh all genes with new papers | Monthly |
| Cell 7 | Sync DB to website | After Cell 4, 5, or 6 |
| Cell 8 | Migrate old cache files to DB | One-time only |
| Cell 9 | View DB status | Anytime |

---
**Before starting:** Runtime → Change runtime type → T4 GPU

In [ ]:
#CELL 1: Install dependencies
%%capture
!pip install biopython pandas transformers accelerate bitsandbytes \
             huggingface-hub google-auth google-auth-httplib2 google-api-python-client
print('Dependencies installed.')

In [ ]:
#Cell 2: Mount Drive + HuggingFace login
from google.colab import drive
drive.mount('/content/drive')

#Hugging Face Token
from huggingface_hub import login
login('hf_YOUR_TOKEN_HERE')

import os
os.makedirs('/content/drive/MyDrive/pubmed_llm/gene_function_lab', exist_ok=True)
print('Drive mounted.')

In [ ]:
# CELL 3: Configure paths, load modules, verify DB
# Run this after Cell 2 every session. Sets up everything.
import os, sys, importlib, json

# Paths (edit if your folder structure is different)
DRIVE_FOLDER = '/content/drive/MyDrive/pubmed_llm'
DB_PATH      = f'{DRIVE_FOLDER}/gene_function_lab/gene_function_lab.db'
SA_KEY_PATH  = f'{DRIVE_FOLDER}/service-account.json'
CACHE_DIR    = '/content/drive/MyDrive/pubmed_llm/functional_study_cache'
YOUR_EMAIL   = 'ssd2184@columbia.edu'

# Load service account key for Drive uploads
if os.path.exists(SA_KEY_PATH):
    with open(SA_KEY_PATH) as f:
        os.environ['GOOGLE_SERVICE_ACCOUNT_JSON'] = f.read()
    print('Service account loaded.')
else:
    print(f'WARNING: service_account.json not found at {SA_KEY_PATH}')
    print('Drive uploads will not work until you add it.')

# Add Drive folder to Python path
if DRIVE_FOLDER not in sys.path:
    sys.path.insert(0, DRIVE_FOLDER)

# Load db and force correct DB path
import db
importlib.reload(db)
db.DB_PATH = DB_PATH
db.init_db()

# Patch pipeline.py with your email
pipeline_path = f'{DRIVE_FOLDER}/pipeline.py'
if os.path.exists(pipeline_path) and YOUR_EMAIL != 'your_email@example.com':
    with open(pipeline_path) as f:
        src = f.read()
    if 'your_email@example.com' in src:
        src = src.replace('your_email@example.com', YOUR_EMAIL)
        with open(pipeline_path, 'w') as f:
            f.write(src)
        print(f'pipeline.py patched with: {YOUR_EMAIL}')

# Load pipeline
import pipeline
importlib.reload(pipeline)

# Show DB status
stats = db.db_stats()
print(f'\nDB path:    {db.DB_PATH}')
print(f'Papers:     {stats["total_papers"]}')
print(f'Genes:      {", ".join(stats["all_genes"]) or "none yet"}')
print(f'Functional: {stats["functional_papers"]}')

pending = db.get_pending_requests()
print(f'\nPending website requests: {len(pending)}')
for r in pending:
    print(f'  {r["gene"]} requested at {r["requested_at"]}')
print('\nReady.')

In [ ]:
# CELL 4: Add a list of genes to the database
# Processes genes directly without going through the website queue. Already-processed genes are automatically skipped.
import importlib, gc, torch
import db, pipeline
importlib.reload(db)
importlib.reload(pipeline)
db.DB_PATH = DB_PATH  # ensure correct path

# Edit this list
GENES_TO_ADD = [
    'TSPAN3'
    # add more genes here
]
MAX_PAPERS = 300

# Check which are new
to_process, already_done = [], []
for gene in GENES_TO_ADD:
    gene = gene.upper().strip()
    if db.gene_is_processed(gene):
        already_done.append(gene)
        print(f'  {gene}: already in DB, skipping')
    else:
        to_process.append(gene)
        print(f'  {gene}: will process')

print(f'\nProcessing {len(to_process)} gene(s), skipping {len(already_done)}.')

# Run pipeline
for i, gene in enumerate(to_process):
    print(f'\n[{i+1}/{len(to_process)}] {gene}')
    try:
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            gc.collect()
        rows = pipeline.analyze_gene(gene, max_papers=MAX_PAPERS)
        if rows:
            db.upsert_papers_bulk(rows)
        db.update_gene_record(gene)
        print(f'  Done: {len(rows)} new papers saved for {gene}.')
    except Exception as e:
        print(f'  Error: {e}')

stats = db.db_stats()
print(f'\nDB now has {stats["total_papers"]} papers across {stats["total_genes"]} genes.')
print('Run Cell 7 to sync to the website.')

In [ ]:
# CELL 5: Process website gene requests
# Run when members have requested genes via the website.
# Processes a bounded batch, saves results to SQLite, uploads the DB, then stops.
import importlib, time, gc, torch
import db, pipeline, drive_sync
importlib.reload(db)
importlib.reload(pipeline)
importlib.reload(drive_sync)
db.DB_PATH = DB_PATH
drive_sync._drive_file_id = None

# Adjust these for your compute budget.
# Start with 3-5 genes per Colab session if using the free/limited GPU runtime.
MAX_REQUESTS_THIS_RUN = 5
MAX_PAPERS_OVERRIDE = None  # set to 50 or 100 for a fast backlog triage run
POLL_WHEN_EMPTY = False
POLL_INTERVAL = 60
RECOVER_STUCK_PROCESSING = True

if RECOVER_STUCK_PROCESSING and hasattr(db, 'reset_processing_requests'):
    reset_n = db.reset_processing_requests()
    if reset_n:
        print(f'Reset {reset_n} interrupted processing request(s) back to pending.')

pending = db.get_pending_requests()
print(f'Pending requests: {len(pending)}')
for r in pending:
    print(f'  {r["gene"]} requested {r["requested_at"]}')
print(f'
Worker starting. This run will process up to {MAX_REQUESTS_THIS_RUN} request(s).
')

processed = 0
while True:
    if MAX_REQUESTS_THIS_RUN is not None and processed >= MAX_REQUESTS_THIS_RUN:
        print(f'Processed {processed} request(s). Stopping to conserve compute.')
        break

    pending = db.get_pending_requests()

    if not pending:
        if not POLL_WHEN_EMPTY:
            print('Queue empty. Stopping.')
            break
        print(f'[{time.strftime("%H:%M:%S")}] Queue empty. Checking again in {POLL_INTERVAL}s...')
        time.sleep(POLL_INTERVAL)
        continue

    req        = pending[0]
    gene       = req['gene']
    qid        = req['id']
    max_papers = MAX_PAPERS_OVERRIDE or req.get('max_papers', 300) or 300

    print(f'
[{time.strftime("%H:%M:%S")}] Processing: {gene} (max {max_papers} papers)')
    db.mark_queue_processing(qid)

    try:
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            gc.collect()

        rows = pipeline.analyze_gene(gene, max_papers=max_papers)
        if rows:
            db.upsert_papers_bulk(rows)
        db.update_gene_record(gene)
        db.mark_queue_done(qid)
        processed += 1
        print(f'  Done: {len(rows)} new papers saved for {gene}.')

        # Auto-upload to Drive after each gene completes
        drive_sync.upload_db_to_drive(DB_PATH)
        print('  DB synced to Drive.')

    except Exception as e:
        processed += 1
        print(f'  Error: {e}')
        db.mark_queue_error(qid, str(e))

    time.sleep(2)


In [ ]:
# CELL 6: Refresh existing genes (monthly, chunked)
# Run monthly. The pipeline skips PMIDs already present in the DB/cache and processes only new hits.
import importlib, time, gc, torch
import db, pipeline, drive_sync
importlib.reload(db)
importlib.reload(pipeline)
importlib.reload(drive_sync)
db.DB_PATH = DB_PATH
drive_sync._drive_file_id = None

# Adjust these for your compute budget. Example: run 10 genes/session, then set START_AT_INDEX=10 next time.
START_AT_INDEX = 0
MAX_GENES_THIS_RUN = 10
MAX_NEW_PAPERS_PER_GENE = 100

stats = db.db_stats()
all_genes = stats['all_genes']
end_index = None if MAX_GENES_THIS_RUN is None else START_AT_INDEX + MAX_GENES_THIS_RUN
genes_to_refresh = all_genes[START_AT_INDEX:end_index]

if not genes_to_refresh:
    print('No genes selected. Check START_AT_INDEX / MAX_GENES_THIS_RUN or add genes first.')
else:
    print(f'Refreshing {len(genes_to_refresh)} gene(s), index {START_AT_INDEX} to {START_AT_INDEX + len(genes_to_refresh) - 1}:')
    print(', '.join(genes_to_refresh))
    print('Already-processed PMIDs are skipped using the DB/cache.
')

    for i, gene in enumerate(genes_to_refresh):
        print(f'[{START_AT_INDEX+i+1}/{len(all_genes)}] {gene}...')
        try:
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
                gc.collect()
            rows = pipeline.analyze_gene(gene, max_papers=MAX_NEW_PAPERS_PER_GENE)
            if rows:
                db.upsert_papers_bulk(rows)
            db.update_gene_record(gene)
            print(f'  {len(rows)} new papers saved.')
        except Exception as e:
            print(f'  Error: {e}')
        time.sleep(1)

    print('
Uploading to Drive...')
    drive_sync.upload_db_to_drive(DB_PATH)

    stats = db.db_stats()
    print('
Refresh chunk complete.')
    print(f'Total papers:  {stats["total_papers"]}')
    print(f'Functional:    {stats["functional_papers"]}')
    print('If more genes remain, rerun this cell with a larger START_AT_INDEX.')


In [ ]:
# CELL 8: Migrate cache files into DB
import glob, json, importlib
import db
importlib.reload(db)
db.DB_PATH = DB_PATH

REMAP = {
    'evidence_functional_study': 'evidence_perturbation',
    'overall_decision':          'llm_reasoning',
}
DROP_COLS = {
    'functional_study_bool', 'in_vitro_bool', 'in_vivo_bool',
    'knockout_bool', 'knockdown_bool', 'shrna_bool', 'sirna_bool',
    'crispr_bool', 'crispr_screen_bool', 'confidence_float',
}

print(f'Scanning: {pipeline.CACHE_DIR}')
files = glob.glob(f'{pipeline.CACHE_DIR}/*.json')
print(f'Found {len(files)} cache files.')

rows, skipped = [], 0
for fp in files:
    try:
        with open(fp) as f:
            data = json.load(f)
        if data.get('_skip') or 'pmid' not in data:
            skipped += 1
            continue
        data['gene'] = str(data.get('gene', '')).upper()
        for old, new in REMAP.items():
            if old in data and new not in data:
                data[new] = data.pop(old)
            elif old in data:
                data.pop(old)
        for col in DROP_COLS:
            data.pop(col, None)
        rows.append(data)
    except:
        continue

print(f'{len(rows)} papers to migrate, {skipped} skipped.')

if rows:
    db.upsert_papers_bulk(rows)
    genes = list(set(r['gene'] for r in rows))
    for gene in genes:
        db.update_gene_record(gene)
    print(f'Migrated {len(rows)} papers for: {", ".join(sorted(genes))}')

stats = db.db_stats()
print(f'\nDB now has {stats["total_papers"]} papers across {stats["total_genes"]} genes.')
print('Run Cell 7 to sync to the website.')

In [ ]:
# CELL 7: Sync DB to the website
# Run after Cell 4, 5, or 6 to push changes live.
# The site also auto-syncs hourly, but this forces an immediate update.
import importlib
import drive_sync
importlib.reload(drive_sync)
drive_sync._drive_file_id = None

print(f'Uploading {DB_PATH} to Drive...')
ok = drive_sync.upload_db_to_drive(DB_PATH)

if ok:
    print('Done. To force the website to refresh immediately, open:')
    print('https://huggingface.co/spaces/shenovad/reyalab/api/stats')
    print('Otherwise the site auto-refreshes within 1 hour.')
else:
    print('Upload failed. Check error above.')
    print('Make sure service-account.json exists in your Drive folder.')

In [ ]:
# CELL 9: View DB status
# Full summary of what is in the database and the request queue.
import importlib
import db
importlib.reload(db)
db.DB_PATH = DB_PATH

stats = db.db_stats()

print('Database')
print(f'  Path:       {db.DB_PATH}')
print(f'  Papers:     {stats["total_papers"]}')
print(f'  Genes:      {stats["total_genes"]}')
print(f'  Functional: {stats["functional_papers"]}')

print('\nPapers per gene')
for g in stats['top_genes']:
    s = db.gene_summary(g['gene'])
    print(f'  {g["gene"]:<12} {s["total"]:>4} total  {s["functional"]:>3} functional  pancreatic:{s["pancreatic"]}  gi:{s["gi"]}')

print('\nCancer type breakdown (functional papers only)')
for ctype, count in stats.get('cancer_type_breakdown', {}).items():
    print(f'  {ctype:<15} {count}')

print('\nRequest queue (last 10)')
queue = db.get_all_queue()
if queue:
    for r in queue[:10]:
        fin = r['finished_at'] or 'not finished'
        print(f'  {r["gene"]:<12} {r["status"]:<12} {r["requested_at"]}  -> {fin}')
else:
    print('  No requests yet.')